# Simon Algorithm — Teaching Notebook

Simon’s algorithm discovers a hidden XOR period. It is the canonical bridge from Deutsch–Jozsa to Shor because both algorithms recover hidden algebraic structure using quantum interference.

This notebook is designed to be **self-explanatory for classroom teaching**. It moves from the problem statement and mathematics to quantum circuits, Qiskit implementation, interpretation, limitations, and exercises.

### Learning objectives
- understand the 2-to-1 hidden-XOR promise
- derive why measurements satisfy y·s=0 mod 2
- build Simon oracles for teaching examples
- recover the secret with GF(2) linear algebra
- compare Simon with classical collision search

## 0. Installation
Run this only if Qiskit is not already installed.

In [ ]:
%pip install -q qiskit qiskit-aer matplotlib numpy

## 1. Imports and helper functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from fractions import Fraction
from math import gcd, pi

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector, Operator
from qiskit_aer import AerSimulator

backend = AerSimulator()

def run_counts(qc, shots=2048):
    compiled = transpile(qc, backend)
    return backend.run(compiled, shots=shots).result().get_counts()

def plot_counts(counts, title='Measurement counts'):
    keys = sorted(counts)
    vals = [counts[k] for k in keys]
    plt.figure(figsize=(8,4))
    plt.bar(keys, vals)
    plt.xlabel('bit string')
    plt.ylabel('counts')
    plt.title(title)
    plt.xticks(rotation=45)
    plt.show()

# Part I — Simon's promise problem
We are given

$$f:\{0,1\}^n\rightarrow\{0,1\}^n$$

with the promise that there exists a secret nonzero string $s$ such that

$$f(x)=f(y)\iff y=x\oplus s.$$

Thus each output has exactly two preimages, separated by the same XOR mask $s$. The goal is to find $s$.

## 2. Example
If $s=110$, then $x$ and $x\oplus110$ always produce the same output. For example, $001$ pairs with $111$.

In [ ]:
s='110'
for x in range(8):
    xb=format(x,'03b')
    paired=''.join(str(int(a)^int(b)) for a,b in zip(xb,s))
    print(xb,'<->',paired)

# Part II — Quantum derivation
Prepare two $n$-qubit registers:

$$|0^n\rangle|0^n\rangle.$$

After Hadamards on the first register,

$$\frac1{\sqrt{2^n}}\sum_x|x\rangle|0\rangle.$$

The oracle gives

$$\frac1{\sqrt{2^n}}\sum_x|x\rangle|f(x)\rangle.$$

Measuring the output register to some $f(x_0)$ leaves the input register in

$$\frac{|x_0\rangle+|x_0\oplus s\rangle}{\sqrt2}.$$

## 3. Second Hadamard layer
Using

$$H^{\otimes n}|x\rangle=\frac1{\sqrt{2^n}}\sum_y(-1)^{x\cdot y}|y\rangle,$$

we obtain amplitudes proportional to

$$(-1)^{x_0\cdot y}\left[1+(-1)^{s\cdot y}\right].$$

This vanishes whenever $s\cdot y=1$ mod 2. Therefore every observed bit string satisfies

$$\boxed{s\cdot y=0\pmod2}.$$

Each run gives one linear equation about $s$.

# Part III — A simple Simon oracle
For teaching we can construct an oracle for a secret whose least-significant set bit is used as a pivot. The output map is linear over GF(2) and has kernel span$(s)$.

In [ ]:
def simon_linear_matrix(secret):
    # secret given as qubit-order bits s[0],...,s[n-1]
    s=np.array(secret,dtype=int)
    n=len(s)
    pivot=np.where(s==1)[0][0]
    rows=[]
    for j in range(n):
        if j==pivot:
            continue
        r=np.zeros(n,dtype=int)
        r[j]=1
        if s[j]: r[pivot]=1
        rows.append(r)
    # one extra zero row keeps n output qubits
    rows.append(np.zeros(n,dtype=int))
    return np.array(rows,dtype=int)

A=simon_linear_matrix([1,1,0])
print(A)
print('A s mod2 =', A@np.array([1,1,0])%2)

## 4. Convert the GF(2) matrix to CNOTs

In [ ]:
def simon_oracle(secret):
    secret=np.array(secret,dtype=int)
    n=len(secret)
    A=simon_linear_matrix(secret)
    qc=QuantumCircuit(2*n,name='Simon oracle')
    for out,row in enumerate(A):
        for inp,b in enumerate(row):
            if b:
                qc.cx(inp,n+out)
    return qc

print(simon_oracle([1,1,0]).draw())

## 5. Full Simon circuit

In [ ]:
def simon_circuit(secret):
    n=len(secret)
    qc=QuantumCircuit(2*n,n)
    qc.h(range(n))
    qc.compose(simon_oracle(secret), inplace=True)
    qc.h(range(n))
    qc.measure(range(n),range(n))
    return qc

qc=simon_circuit([1,1,0])
print(qc.draw())
counts=run_counts(qc,4096)
print(counts)

## 6. Verify the orthogonality condition
Qiskit displays classical bit strings in reverse qubit order, so convert each measurement back to qubit order before taking the dot product.

In [ ]:
secret=np.array([1,1,0],dtype=int)
for bitstr,c in sorted(counts.items()):
    y=np.array([int(b) for b in bitstr[::-1]])
    print(bitstr, 'qubit-order y=',y, 'y·s mod2=', int(y@secret%2), 'count=',c)

# Part IV — Solve equations over GF(2)

In [ ]:
def gf2_rank(M):
    A=np.array(M,dtype=np.uint8).copy()%2
    r=0
    for c in range(A.shape[1]):
        piv=next((i for i in range(r,A.shape[0]) if A[i,c]),None)
        if piv is None: continue
        A[[r,piv]]=A[[piv,r]]
        for i in range(A.shape[0]):
            if i!=r and A[i,c]: A[i]^=A[r]
        r+=1
    return r

def recover_secret_bruteforce(equations):
    Y=np.array(equations,dtype=int)%2
    n=Y.shape[1]
    candidates=[]
    for x in range(1,2**n):
        s=np.array([(x>>q)&1 for q in range(n)],dtype=int)
        if np.all(Y@s%2==0): candidates.append(s)
    return candidates

eqs=[]
for bitstr in counts:
    y=np.array([int(b) for b in bitstr[::-1]])
    if np.any(y) and gf2_rank(eqs+[y])>gf2_rank(eqs):
        eqs.append(y)
    if gf2_rank(eqs)>=2: break
print('independent equations:',eqs)
print('secret candidates:',recover_secret_bruteforce(eqs))

# Part V — Complexity
Quantumly, $O(n)$ successful samples are enough to collect about $n-1$ independent equations, followed by polynomial-time linear algebra. Classical randomized algorithms need roughly $\Theta(2^{n/2})$ oracle queries to find a collision and infer the hidden XOR mask.

## 8. Why Simon matters historically
Simon provided one of the first exponential quantum query separations and directly influenced Shor's algorithm. Both can be understood as **hidden subgroup / hidden periodicity** problems:

- Simon: period under XOR in $(\mathbb Z_2)^n$;
- Shor: period under addition in the integers modulo an order.

# Part VI — Common misconceptions
- One quantum measurement does not reveal $s$ directly.
- Every sample is a constraint $y\cdot s=0$, not a bit of $s$.
- Multiple independent runs are necessary.
- The promise that all collisions are generated by the same $s$ is essential.
- The second register can be ignored after it has created the entanglement/correlation structure; explicitly measuring it is conceptually useful but not required for the final distribution.

# Part VII — Exercises
1. For $s=101$, list all $y$ satisfying $y\cdot s=0$.
2. Show that the set of valid $y$ forms an $(n-1)$-dimensional subspace.
3. Change the code to secret 1011.
4. Explain why $n-1$ independent equations leave one nonzero vector in the null space.
5. Compare Simon's XOR period with Shor's arithmetic period.

# Part VIII — Solutions
For $s=101$, the condition is $y_0+y_2=0\pmod2$, so $y_0=y_2$. With three bits in qubit order the valid vectors are 000, 010, 101, and 111. The valid measurement outcomes are exactly the orthogonal complement of the one-dimensional subspace spanned by $s$.

# Compact reference
$$f(x)=f(x\oplus s)$$

Quantum sampling gives

$$y\cdot s=0\pmod2.$$

Collect $n-1$ independent equations and solve

$$Ys=0\pmod2$$

for the unique promised nonzero null vector $s$.